<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 — Exercise: Store and Search with ChromaDB

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Do

Nine short steps. The installs, the imports and the text are given — **you write every other cell.**

```
6 sentences  ->  embed  ->  Chroma collection  ->  ask  ->  top 2 + distances
a document   ->  chunk  ->  embed  ->  Chroma collection  ->  ask
```

1. **Embed** six sentences with a free local model
2. **Store** them in an in-memory Chroma collection
3. **Ask** three questions and read the distances
4. Do the whole thing again on a **real document** — chunked first
5. Ask it something it has never heard of, and see what comes back anyway

> **No API key needed.** Every cell runs locally on the Colab CPU.


---

## Setup

Run these three cells. Nothing to write yet — the second one downloads the embedding model
(about a minute).

In [ ]:
# PROVIDED - just run this cell.
!pip install -q chromadb sentence-transformers langchain-text-splitters

In [ ]:
# PROVIDED - just run this cell.
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imports ready")

### The text you'll use

Six sentences covering three topics, and one longer document for the second half.

In [ ]:
# PROVIDED - just run this cell.

# Three topics, two sentences each - space, cooking, sport.
SENTENCES = [
    "ISRO's Chandrayaan-3 touched down near the Moon's south pole.",
    "The James Webb telescope photographs distant galaxies in infrared light.",
    "Soak the rice for twenty minutes before you start the biryani.",
    "Add the tempering of mustard seeds and curry leaves right at the end.",
    "India chased down 280 runs with two balls to spare.",
    "The striker scored a hat-trick in the second half.",
]

# A campus handbook - you will chunk this one in Step 6.
DOCUMENT = """The central library is open from 8 am to 10 pm on weekdays and from 9 am to 6 pm on Saturdays.
It stays closed on Sundays and gazetted holidays. Each student may borrow four books at a time for
fourteen days, renewable twice online. A fine of five rupees per day applies to overdue titles, and
reference volumes may not be taken out of the reading hall.

The mess follows a weekly rotating menu and serves breakfast from 7:30 to 9:30, lunch from 12:30 to
2:30 and dinner from 7:30 to 9:30. Hostel gates close at 10 pm, and residents returning later must
record an entry with the warden on duty. Rooms are allotted for the full academic year and cannot be
exchanged after the first month without written approval.

End-semester examinations are held in the first two weeks of December and May. A student needs
seventy-five percent attendance in a course to be allowed to sit for its paper. Electronic devices of
any kind are prohibited inside the examination hall, and a re-evaluation request must be filed within
ten days of the result being published.

The sports complex houses a gymnasium, two badminton courts and a floodlit football ground. The gym
is open from 6 am to 9 am and again from 5 pm to 9 pm. Equipment for cricket, basketball and table
tennis is issued from the sports office against a student ID card."""

print(len(SENTENCES), "sentences and", len(DOCUMENT), "characters ready")

---

## Step 1 — Embed the six sentences

`all-MiniLM-L6-v2` runs locally on the Colab CPU. Free, fast, no key.

In [ ]:
# 1. load SentenceTransformer('all-MiniLM-L6-v2')  ->  model
# 2. encode SENTENCES  ->  sentence_embeddings
# 3. print sentence_embeddings.shape


# your code here

**Expect `(6, 384)`** — six rows, one per sentence, 384 numbers each.

---

## Step 2 — Create an in-memory collection

`chromadb.Client()` keeps everything in RAM. It disappears when the runtime restarts, which is
exactly what you want for an exercise.

In [ ]:
# 1. build the client  ->  chromadb.Client()
# 2. make a collection named "warmup"  ->  get_or_create_collection(name=...)


# your code here

---

## Step 3 — Add the sentences

Chroma wants three lists of the same length, side by side.

In [ ]:
# ids        -> one unique string per sentence: "s0", "s1", ...
# embeddings -> sentence_embeddings.tolist()   (a plain list, not a numpy array)
# documents  -> SENTENCES                      (so you get the text back, not a row number)
#
# then print collection.count()


# your code here

---

## Step 4 — Ask a question

Embed the question **with the same model**, then hand that vector to `query()`.
Notice the question shares almost no words with the sentence that should come back.

In [ ]:
query = "Which spacecraft reached the Moon?"

# 1. encode [query]        -> note the list: one query in, one vector out
# 2. collection.query(query_embeddings=..., n_results=2)
# 3. print the query, then each of the 2 documents with its distance
#    they live at  results['documents'][0]  and  results['distances'][0]


# your code here

---

## Step 5 — Two more questions

**Change `query` in the cell above and re-run it.** One at a time:

* `"How should I prepare the grains first?"`
* `"Who scored three goals?"`

Nothing in the list says *grains* or *goals*. Did the right sentence still come back first?

⚠️ Two things that trip everyone up:

1. Chroma returns **`distances`, not similarities** — **lower is closer**.
2. `results['documents']` is a **list of lists**. `query()` accepts a *batch* of queries, so your
   answers sit at index `[0]`.

---

## Step 6 — Now a real document: chunk it

One vector cannot represent a page that covers library hours, mess timings, exam rules *and* the
gym all at once. Cut it up first.

In [ ]:
# 1. RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
# 2. .split_text(DOCUMENT)  ->  chunks
# 3. print how many chunks you got, then print chunks[0]


# your code here

**Look at chunk 0.** It should end on a whole word, not halfway through one.

**Now print the short ones.** Two chunks come out tiny — leftover tails from paragraphs that ran
just past `chunk_size`. Read one on its own: could it answer anything at all? That is the price of
a `chunk_size` that doesn't line up with your document.

Then set `chunk_size=80`, re-run, and look again — is a single 80-character chunk ever enough to
answer a question? Put it back to `300` before you go on.

---

## Step 7 — Embed the chunks and store them

The same three lists as Step 3, on a **new** collection so the six sentences stay out of the way.

In [ ]:
# 1. encode chunks  ->  chunk_embeddings
# 2. a new collection named "handbook"
# 3. add it:  ids like "chunk_0", "chunk_1", ...  ·  embeddings .tolist()  ·  documents=chunks
# 4. print the count


# your code here

---

## Step 8 — Search the handbook

The same two moves as Step 4, pointed at the new collection.

In [ ]:
question = "When can I borrow books?"        # <-- change this and re-run

# encode the question, query the handbook for the top 2,
# and print each chunk with its distance.
#
# Try these too:  "what time is dinner?"
#                 "how much attendance do I need?"
#                 "where do I get a cricket bat?"


# your code here

---

## Step 9 — Break it on purpose

Ask the handbook something it has never heard of.

In [ ]:
question = "how much does a hostel room in Dubai cost?"

# run the same search as Step 8 with this question.
# it will still hand you 2 chunks, confidently. look at the distances -
# how do they compare with the ones you got in Step 8?


# your code here

**Similarity search always returns its top-k.** It has no idea that nothing is relevant — the only
clue you get is that the distances are *worse than usual*. Where to draw that line is Day 4's job.

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **`SentenceTransformer.encode()`** | text → 384 numbers, local and free, no key |
| **`chromadb.Client()`** | an in-memory vector store — gone when the runtime restarts |
| **`collection.add()`** | three parallel lists: `ids`, `embeddings` (`.tolist()`), `documents` |
| **`collection.query()`** | `query_embeddings` + `n_results` → the nearest k |
| **`distances`** | Chroma gives distance, not similarity — **lower is closer** |
| **The `[0]`** | results are lists of lists, because `query()` takes a batch of queries |
| **Same model both sides** | query and documents must be embedded by the *same* model |
| **Chunk first** | a whole document is one blurred vector — split it, then embed the pieces |
| **Top-k always answers** | ask about something absent and you still get k chunks back |
